<a href="https://colab.research.google.com/github/davesagit123/blank-app/blob/main/benchmark_finder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11"
  }
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Sectional Repeatability Filter \u2014 One-Click Colab Notebook\n",
    "",
    "**Why a one-cell version?** When a notebook is opened, its cells are EMPTY (`execution_count: null`) until you run them \u2014 Colab does NOT auto-execute. This file bundles EVERYTHING into ONE cell, so a single click produces the result.\n",
    "",
    "**How to use:**\n",
    "1. In Colab: File \u2192 Upload \u2192 upload `horse_sectional_filter_oneclick.ipynb` (or create a new Colab notebook and paste this cell's content).\n",
    "2. OPTIONAL STEP 1: Replace the data block between `### USER INPUT DATA START/END` with your own rows.\n",
    "3. CLICK RUN ON THE CELL (or Kernel \u2192 Restart & Run All).\n",
    "",
    "**Data format** (one runner per line): `EntryNum  HorseName  L800m  L600m  L400m  L200m`, signed decimals allowed; trailing `L` stripped automatically. Blank lines / `#` comments / lines containing `scratch` are skipped."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# --------------------------------------------------------------------------\n",
    "### USER INPUT DATA START\n",
    "# Paste your race page here (one runner per line):\n",
    "#   <entry #>  <horse name>  L800m  L600m  L400m  L200m\n",
    "# Signed decimals allowed; trailing 'L' stripped automatically.\n",
    "# Blank lines / '#' comments / rows containing 'scratch' are ignored.\n",
    "sectionals = '''\n",
    "1  The Torque  -0.75L  -0.99L  -0.89L  -0.33L\n",
    "2  Tsunuki  +0.03L  -0.61L  -0.26L  +0.45L\n",
    "3  Bee Exact  -0.14L  -0.40L  -0.38L  +0.02L\n",
    "5  Change Th  -0.32L  -0.46L  -0.37L  -0.18L\n",
    "6  Clubhouse  -0.75L  -0.85L  -0.66L  -0.24L\n",
    "7  Switchblac  +0.42L  +0.63L  +0.35L  +0.15L\n",
    "10  Brutal Glor  -0.18L  -0.51L  -0.64L  -0.42L\n",
    "11  Shining Prc  +0.61L  +0.16L  +0.17L  +0.37L\n",
    "13  Ed I Am  +0.02L  -0.05L  -0.07L  +0.31L\n",
    "15  Save The F  +0.17L  +0.04L  -0.01L  +0.09L\n",
    "16  So You Car  -0.34L  -0.12L  +0.24L  +0.39L\n",
    "'''\n",
    "### USER INPUT DATA END\n",
    "# --------------------------------------------------------------------------\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "\n",
    "def _to_float(tok):\n",
    "    t = tok.strip().replace(\"L\", \"\").strip()\n",
    "    try:\n",
    "        return float(t)\n",
    "    except ValueError:\n",
    "        return np.nan\n",
    "\n",
    "rows = []\n",
    "for line in sectionals.strip().splitlines():\n",
    "    line = line.strip()\n",
    "    if not line or line.startswith(\"#\"):\n",
    "        continue\n",
    "    toks = line.split()\n",
    "    if len(toks) < 6:\n",
    "        continue\n",
    "    name = \" \".join(toks[1:-4])\n",
    "    if \"scratch\" in name.lower():\n",
    "        continue\n",
    "    try:\n",
    "        entry = int(toks[0])\n",
    "    except ValueError:\n",
    "        continue\n",
    "    nums = [_to_float(t) for t in toks[-4:]]\n",
    "    if any(np.isnan(x) for x in nums):\n",
    "        continue\n",
    "    rows.append((entry, name, *nums))\n",
    "\n",
    "df = pd.DataFrame(rows, columns=[\"entry\",\"horse\",\"L800m\",\"L600m\",\"L400m\",\"L200m\"])\n",
    "n_active = len(df)\n",
    "print(f\"Active runners parsed : n = {n_active}\")\n",
    "if n_active >= 2:\n",
    "    missing = sorted(set(range(df['entry'].min(), df['entry'].max() + 1)) - set(df['entry']))\n",
    "    print(f\"Already-scratched entry numbers detected: {missing or 'NONE'}\")\n",
    "\n",
    "FUNNEL_COLS = [\"L800m\", \"L600m\", \"L400m\", \"L200m\"]\n",
    "SPLITS = [\"L600m\", \"L400m\", \"L200m\"]\n",
    "SHORT = {\"L600m\": \"L600\", \"L400m\": \"L400\", \"L200m\": \"L200\"}\n",
    "funnel = {}\n",
    "for col in FUNNEL_COLS:\n",
    "    mean, sigma = df[col].mean(), df[col].std(ddof=0)\n",
    "    bench = mean - 1.645 * sigma\n",
    "    funnel[col] = (mean, sigma, bench)\n",
    "print(\"\\nFUNNEL (benchmark = mean - 1.645*std, ddof=0):\")\n",
    "for col, sh in SHORT.items():\n",
    "    m, s, b = funnel[col]\n",
    "    print(f\"  {sh}: benchmark = {b:+.2f}  (mean {m:+.2f}, std {s:+.2f})\")\n",
    "\n",
    "best_runners, Z, warn = {}, {}, []\n",
    "print(\"\\nBEST RUNNER per split (lower = faster):\")\n",
    "for col in SPLITS:\n",
    "    mean, sigma, bench = funnel[col]\n",
    "    idx = df[col].idxmin()\n",
    "    horse, val = df.loc[idx, 'horse'], df.loc[idx, col]\n",
    "    z = (val - mean) / sigma\n",
    "    best_runners[col] = (horse, val); Z[col] = z\n",
    "    print(f\"  {SHORT[col]}: {horse} @ {val:+.2f}L  Z = {z:+.2f}\")\n",
    "    if z > 0:\n",
    "        warn.append((SHORT[col], horse, val, z))\n",
    "print(\"Direction guardrail:\", \"PASS (all best-runner Z negative)\" if not warn else \"FLAGGED (positive Z = slow side)!\")\n",
    "\n",
    "print('\\n' + '='*44)\n",
    "print('FORM-PAGE SCORING OUTPUT (exact format)')\n",
    "print('='*44)\n",
    "for col in SPLITS:\n",
    "    h, v = best_runners[col]\n",
    "    print(f\"{SHORT[col]:<7} Benchmark        {funnel[col][2]:+.2f}\")\n",
    "    print(f\"{SHORT[col]:<7} Best Runner Z-Score {Z[col]:+.2f} ({h}, {v:+.2f}L)\")\n",
    "print('='*44)\n",
    "\n",
    "print(\"\\nBEAT CHECK (value <= benchmark, lower-is-better):\")\n",
    "split_beaters = {}\n",
    "for col in SPLITS:\n",
    "    _, _, bench = funnel[col]\n",
    "    hits = df[df[col] <= bench]\n",
    "    split_beaters[col] = set(hits['horse'])\n",
    "    if hits.empty:\n",
    "        print(f\"  {SHORT[col]}: NONE below benchmark {bench:+.2f} (field best = {df[col].min():+.2f}L)\")\n",
    "    else:\n",
    "        print(f\"  {SHORT[col]}: {sorted(hits['horse'])} beat it\")\n",
    "common = set.intersection(*split_beaters.values()) if split_beaters else set()\n",
    "print()\n",
    "if common:\n",
    "    print(f\">>> HORSES beating ALL THREE benchmarks: {sorted(common)}\")\n",
    "else:\n",
    "    print(\">>> NO horse beats all three benchmarks on this page.\")\n"
   ]
  }
 ]
}

# Sectional Repeatability Filter — One-Click Colab Notebook

**Why a one-cell version?** When a notebook is opened, its cells are EMPTY (`execution_count: null`) until you run them — Colab does NOT auto-execute. This file bundles EVERYTHING into ONE cell, so a single click produces the result.

**How to use:**
1. In Colab: File → Upload → upload `horse_sectional_filter_oneclick.ipynb` (or create a new Colab notebook and paste this cell's content).
2. OPTIONAL STEP 1: Replace the data block between `### USER INPUT DATA START/END` with your own rows.
3. CLICK RUN ON THE CELL (or Kernel → Restart & Run All).

**Data format** (one runner per line): `EntryNum  HorseName  L800m  L600m  L400m  L200m`, signed decimals allowed; trailing `L` stripped automatically. Blank lines / `#` comments / lines containing `scratch` are skipped.

In [2]:
# --------------------------------------------------------------------------
### USER INPUT DATA START
# Paste your race page here (one runner per line):
#   <entry #>  <horse name>  L800m  L600m  L400m  L200m
# Signed decimals allowed; trailing 'L' stripped automatically.
# Blank lines / '#' comments / rows containing 'scratch' are ignored.
sectionals = '''
1  The Torque  -0.75L  -0.99L  -0.89L  -0.33L
2  Tsunuki  +0.03L  -0.61L  -0.26L  +0.45L
3  Bee Exact  -0.14L  -0.40L  -0.38L  +0.02L
5  Change Th  -0.32L  -0.46L  -0.37L  -0.18L
6  Clubhouse  -0.75L  -0.85L  -0.66L  -0.24L
7  Switchblac  +0.42L  +0.63L  +0.35L  +0.15L
10  Brutal Glor  -0.18L  -0.51L  -0.64L  -0.42L
11  Shining Prc  +0.61L  +0.16L  +0.17L  +0.37L
13  Ed I Am  +0.02L  -0.05L  -0.07L  +0.31L
15  Save The F  +0.17L  +0.04L  -0.01L  +0.09L
16  So You Car  -0.34L  -0.12L  +0.24L  +0.39L
'''
### USER INPUT DATA END
# --------------------------------------------------------------------------

import pandas as pd
import numpy as np

def _to_float(tok):
    t = tok.strip().replace("L", "").strip()
    try:
        return float(t)
    except ValueError:
        return np.nan

rows = []
for line in sectionals.strip().splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    toks = line.split()
    if len(toks) < 6:
        continue
    name = " ".join(toks[1:-4])
    if "scratch" in name.lower():
        continue
    try:
        entry = int(toks[0])
    except ValueError:
        continue
    nums = [_to_float(t) for t in toks[-4:]]
    if any(np.isnan(x) for x in nums):
        continue
    rows.append((entry, name, *nums))

df = pd.DataFrame(rows, columns=["entry","horse","L800m","L600m","L400m","L200m"])
n_active = len(df)
print(f"Active runners parsed : n = {n_active}")
if n_active >= 2:
    missing = sorted(set(range(df['entry'].min(), df['entry'].max() + 1)) - set(df['entry']))
    print(f"Already-scratched entry numbers detected: {missing or 'NONE'}")

FUNNEL_COLS = ["L800m", "L600m", "L400m", "L200m"]
SPLITS = ["L600m", "L400m", "L200m"]
SHORT = {"L600m": "L600", "L400m": "L400", "L200m": "L200"}
funnel = {}
for col in FUNNEL_COLS:
    mean, sigma = df[col].mean(), df[col].std(ddof=0)
    bench = mean - 1.645 * sigma
    funnel[col] = (mean, sigma, bench)
print("\nFUNNEL (benchmark = mean - 1.645*std, ddof=0):")
for col, sh in SHORT.items():
    m, s, b = funnel[col]
    print(f"  {sh}: benchmark = {b:+.2f}  (mean {m:+.2f}, std {s:+.2f})")

best_runners, Z, warn = {}, {}, []
print("\nBEST RUNNER per split (lower = faster):")
for col in SPLITS:
    mean, sigma, bench = funnel[col]
    idx = df[col].idxmin()
    horse, val = df.loc[idx, 'horse'], df.loc[idx, col]
    z = (val - mean) / sigma
    best_runners[col] = (horse, val); Z[col] = z
    print(f"  {SHORT[col]}: {horse} @ {val:+.2f}L  Z = {z:+.2f}")
    if z > 0:
        warn.append((SHORT[col], horse, val, z))
print("Direction guardrail:", "PASS (all best-runner Z negative)" if not warn else "FLAGGED (positive Z = slow side)!")

print('\n' + '='*44)
print('FORM-PAGE SCORING OUTPUT (exact format)')
print('='*44)
for col in SPLITS:
    h, v = best_runners[col]
    print(f"{SHORT[col]:<7} Benchmark        {funnel[col][2]:+.2f}")
    print(f"{SHORT[col]:<7} Best Runner Z-Score {Z[col]:+.2f} ({h}, {v:+.2f}L)")
print('='*44)

print("\nBEAT CHECK (value <= benchmark, lower-is-better):")
split_beaters = {}
for col in SPLITS:
    _, _, bench = funnel[col]
    hits = df[df[col] <= bench]
    split_beaters[col] = set(hits['horse'])
    if hits.empty:
        print(f"  {SHORT[col]}: NONE below benchmark {bench:+.2f} (field best = {df[col].min():+.2f}L)")
    else:
        print(f"  {SHORT[col]}: {sorted(hits['horse'])} beat it")
common = set.intersection(*split_beaters.values()) if split_beaters else set()
print()
if common:
    print(f">>> HORSES beating ALL THREE benchmarks: {sorted(common)}")
else:
    print(">>> NO horse beats all three benchmarks on this page.")

Active runners parsed : n = 11
Already-scratched entry numbers detected: [4, 8, 9, 12, 14]

FUNNEL (benchmark = mean - 1.645*std, ddof=0):
  L600: benchmark = -1.03  (mean -0.29, std +0.45)
  L400: benchmark = -0.86  (mean -0.23, std +0.38)
  L200: benchmark = -0.43  (mean +0.06, std +0.30)

BEST RUNNER per split (lower = faster):
  L600: The Torque @ -0.99L  Z = -1.56
  L400: The Torque @ -0.89L  Z = -1.72
  L200: Brutal Glor @ -0.42L  Z = -1.61
Direction guardrail: PASS (all best-runner Z negative)

FORM-PAGE SCORING OUTPUT (exact format)
L600    Benchmark        -1.03
L600    Best Runner Z-Score -1.56 (The Torque, -0.99L)
L400    Benchmark        -0.86
L400    Best Runner Z-Score -1.72 (The Torque, -0.89L)
L200    Benchmark        -0.43
L200    Best Runner Z-Score -1.61 (Brutal Glor, -0.42L)

BEAT CHECK (value <= benchmark, lower-is-better):
  L600: NONE below benchmark -1.03 (field best = -0.99L)
  L400: ['The Torque'] beat it
  L200: NONE below benchmark -0.43 (field best = -0.42L